## Objective

- The objective of this notebook it to prepare the dataset for the demand forecasting stage.
- In this step, the transactional sales data will be transformed into a structured time series format, making it suitable for future forecasting models. This includes organizing sales by date and product, aggregating demand, and preparing the base dataset for the next stages of the project.
- This notebook focuses **only on data preparation**. Feature engeniring and model training will be handled separately

In [14]:
import os 
from pathlib import Path
import pandas as pd
ROOT = Path("..").resolve()
os.chdir(ROOT)

DATA_PATH = Path("data/raw/motoretail.csv")

df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\motoretail.csv'

## Forecasting Problem

| Item             | Definition                |
| ---------------- | ------------------------- |
| Target           | `quantity_sold`           |
| Forecast horizon | 14 days                   |
| Granularity      | 1 row = 1 product per day |

This granularity makes sense because the goal is to forecast product-level demand over time. By using one row per product per day, the dataset keeps enough detail to support inventory decisions while still creating a clear time series structure for the forecasting model.

## Remove Irrelevant Columns
- The columns bellow will be removed because they will not be necessary for the forecasting analysis.

In [ ]:
df = df.drop(columns=["customer_city", "customer_state", "customer_neighborhood", "customer_address", "customer_zipcode", "sale_time"])
df.head()

,sale_date,day_of_week,month,customer_type,product_name,product_category,quantity_sold,unit_cost_brl,unit_price_brl,discount_pct,...,estimated_profit_brl,payment_method,sales_channel,weather_condition,delivery_app_peak,is_weekend,is_holiday_campaign,current_stock_snapshot,supplier_lead_time_days,returned
0,2025-01-01,Quarta-feira,Janeiro,Novo,Capacete Pro Tork,Capacete,1,180,333.75,0,...,153.75,Pix,Loja Física,Ensolarado,True,False,False,102,10,False
1,2025-01-01,Quarta-feira,Janeiro,Recorrente,Luva Motoboy,Proteção,2,25,77.39,0,...,104.78,Pix,Instagram,Ensolarado,False,False,False,23,9,False
2,2025-01-01,Quarta-feira,Janeiro,Recorrente,Retrovisor Esportivo,Peça,1,35,119.85,0,...,84.85,Pix,Loja Física,Ensolarado,True,False,False,54,9,False
3,2025-01-01,Quarta-feira,Janeiro,Novo,Suporte Celular Moto,Suporte,1,18,60.55,0,...,42.55,Pix,Instagram,Ensolarado,False,False,False,47,7,False
4,2025-01-01,Quarta-feira,Janeiro,Novo,Capacete Pro Tork,Capacete,1,180,324.42,0,...,144.42,Pix,WhatsApp,Ensolarado,True,False,False,93,8,False


## Check and Fix Data Types

In [ ]:
df["sale_date"] = pd.to_datetime(df["sale_date"])

# Checking numeric columns types
df[["quantity_sold", "unit_cost_brl", "unit_price_brl", "discount_pct", "total_revenue_brl", "estimated_profit_brl", "current_stock_snapshot"]].dtypes

# Turn the int columns into float columns
df[["quantity_sold", "unit_cost_brl", "discount_pct", "current_stock_snapshot"]] = df[["quantity_sold", "unit_cost_brl", "discount_pct", "current_stock_snapshot"]].astype(float)

# Checking category columns types
df[["product_category", "payment_method", "sales_channel", "weather_condition", "day_of_week", "month", "product_name",]].dtypes

# Turn the object columns into category columns
df[["product_category", "payment_method", "sales_channel", "weather_condition", "day_of_week", "month", "product_name"]] = df[["product_category", "payment_method", "sales_channel", "weather_condition", "day_of_week", "month", "product_name"]].astype("category")

# Checking the "Ture/False" columns
df[["delivery_app_peak", "is_weekend", "is_holiday_campaign", "returned"]].dtypes



delivery_app_peak      bool
is_weekend             bool
is_holiday_campaign    bool
returned               bool
dtype: object

## Aggregate Sales by Product and Date

In [ ]:
df_agg = (df
    .groupby(["product_name", "sale_date"])
    .agg(
       quantity_sold=("quantity_sold", "sum"),
       total_revenue_brl=("total_revenue_brl", "sum"),
       estimated_profit_brl=("estimated_profit_brl", "sum"),
       discount_pct=("discount_pct", "mean"),
       unit_price_brl=("unit_price_brl", "mean"),
       current_stock_snapshot=("current_stock_snapshot", "mean"),
       weather_condition=("weather_condition", "last"),
       product_category=("product_category", "last"),
    )
    .sort_values("sale_date", ascending=True)
    .reset_index()
)

C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_21916\1960690991.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["product_name", "sale_date"])


## Check the Aggregated Dataset

In [ ]:
df_agg.head()
print("shape:", df_agg.shape)
print(f"Number of duplicated rows: {df_agg.duplicated().sum()}")

shape: (6168, 9)
Number of duplicated rows: 0


## Check Missing Dates

In [16]:
full_grid = pd.MultiIndex.from_product(
    [
        df_agg["product_name"].unique(),
        pd.date_range(df_agg["sale_date"].min(), df_agg["sale_date"].max(), freq="D")
    ],
    names=["product_name", "sale_date"]
).to_frame(index=False)

df_full_check = full_grid.merge(
    df_agg[["product_name", "sale_date", "quantity_sold"]],
    on=["product_name", "sale_date"],
    how="left"
)

missing_product_days = df_full_check[df_full_check["quantity_sold"].isna()]

missing_days_by_product = (
    missing_product_days
    .groupby("product_name", as_index=False)
    .agg(missing_days=("sale_date", "count"))
    .sort_values(by="missing_days", ascending=False)
)

missing_days_by_product

C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_21916\2662122097.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("product_name", as_index=False)


,product_name,missing_days
0,Bag Delivery 45L,0
1,Bag Delivery 80L,0
2,Balaclava,0
3,Baú Moto 30L,0
4,Capa de Chuva,0
5,Capacete LS2,0
6,Capacete Pro Tork,0
7,Carregador USB Moto,0
8,Intercomunicador,0
9,Luva Motoboy,0


## Validate Final Dataset

In [ ]:
missing_pct = df.isna().mean().mul(100).round(2).to_frame("pct_missing")
missing_pct

,pct_missing
sale_date,0.0
day_of_week,0.0
month,0.0
customer_type,0.0
product_name,0.0
product_category,0.0
quantity_sold,0.0
unit_cost_brl,0.0
unit_price_brl,0.0
discount_pct,0.0


In [ ]:
print(f"Number of duplicated rows: {df_agg.duplicated(subset=["sale_date", "product_name"]).sum()}")
print(f"Date range: {df_agg['sale_date'].min()} to {df_agg['sale_date'].max()}")
print(f"Quantity of products: {df["product_name"].nunique()}")
invalid_discount = (df["discount_pct"] > 100).sum()
print(f"Inconsistent values in discount_pct column: {invalid_discount}")

float_cols = df_agg.select_dtypes(include="float").columns
negative_values = (df_agg[float_cols] < 0).sum()

print("Negative float columns:")
negative_values

Number of duplicated rows: 0
Date range: 2025-01-01 00:00:00 to 2026-05-29 00:00:00
Quantity of products: 12
Inconsistent values in discount_pct column: 0
Negative float columns:


quantity_sold             0
total_revenue_brl         0
estimated_profit_brl      0
discount_pct              0
unit_price_brl            0
current_stock_snapshot    0
dtype: int64

## Saving the Processed Dataset

In [ ]:
agg_path = ROOT / "data" / "processed" / "daily_product_sales.csv"
agg_path.parent.mkdir(parents=True, exist_ok=True)
df_agg.to_csv(agg_path, index=False, encoding="utf-8")
print("Saved the processed dataset to:", agg_path.resolve())

Saved the processed dataset to: C:\DEV\motostock-ai\data\processed\daily_product_sales.csv


## Data Preparation Summary

* The transactional sales data was transformed into a structured time series format.
* The dataset was aggregated into a **daily product-level dataset**, grouped by `product_name` and `sale_date`.
* Each row now represents **one product per day**, which fits the forecasting objective of predicting product-level demand.
* The main changes were:

  * Removed unnecessary columns.
  * Applied the correct data types for each column.
  * Aggregated sales, revenue and profit metrics by product and date.
  * Prepared the processed dataset to be used in the forecasting phase.
* The next step is to use the processed data to build a baseline forecasting model.
